# Cascade Denoised Smoothing on Kaggle (CIFAR-10)
Settings: **Accelerator = GPU T4 x2**, **Internet = On**, Persistence = *Files only* (if available).

Each phase can be run in its own session/commit. Outputs go to `/kaggle/working/dds/out`.
To resume after a session ends: add this notebook's previous **Output** as an input and set `PREV_OUT` below.

In [ ]:
# ---------------- config ----------------
SIGMA      = 0.25
SKIP       = 20          # certify images idx % 20 == 0 (500 imgs); probe uses the other 9,500
MAX_IMAGES = 100         # start with 100, later raise to 500 (runs resume, nothing is recomputed)
N          = 10000
BATCH      = 500         # lower to 250 on OOM
AMP        = True        # fp16 on the expensive path; checked in cell 3; SAME flag for dds & cascade
DELTA, CAL_ALPHA = 0.01, 0.05
PREV_OUT   = None        # e.g. "/kaggle/input/<this-notebook-output>/dds/out" to resume
REPO       = "https://github.com/ducquan19/dds.git"
COHEN_GDRIVE_ID = "1h_TpbXm5haY5f-l4--IKylmdz6tvPoR4"   # models.tar from locuslab/smoothing README

import os, subprocess, torch
SIG = f"{SIGMA:.2f}"
AMPF = "--amp" if AMP else ""
NGPU = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(NGPU)])

In [ ]:
# ---------------- 1. setup (~5 min) ----------------
%cd /kaggle/working
if not os.path.exists("dds"):
    !git clone -q {REPO} dds
%cd /kaggle/working/dds
!git pull -q
!pip install -q gdown statsmodels "transformers>=4.40,<5"

if not os.path.exists("cifar10/cifar10_uncond_50M_500K.pt"):
    !wget -q -O cifar10/cifar10_uncond_50M_500K.pt https://openaipublic.blob.core.windows.net/diffusion/march-2021/cifar10_uncond_50M_500K.pt
if not os.path.exists("cifar10/archs"):
    !git clone -q --depth 1 https://github.com/locuslab/smoothing.git /tmp/cohen && cp -r /tmp/cohen/code/archs cifar10/archs
if not os.path.exists("models/cifar10/resnet110/noise_0.25/checkpoint.pth.tar"):
    !gdown -q {COHEN_GDRIVE_ID} -O /tmp/models.tar && mkdir -p /tmp/cm && tar -xf /tmp/models.tar -C /tmp/cm && rm /tmp/models.tar
    !mkdir -p models && cp -r "$(dirname "$(dirname "$(find /tmp/cm -type d -path '*cifar10/resnet110/noise_0.25' | head -1)")")" models/cifar10 && rm -rf /tmp/cm
!ls models/cifar10/resnet110
os.makedirs("out", exist_ok=True); os.makedirs("logs", exist_ok=True)
if PREV_OUT:
    !cp -n {PREV_OUT}/* out/ 2>/dev/null; ls out
import sys; sys.path.insert(0, "cifar10")
from torchvision import datasets
datasets.CIFAR10("data/cifar10", train=False, download=True)
CKPT = f"models/cifar10/resnet110/noise_{SIG}/checkpoint.pth.tar"

In [ ]:
# ---------------- 2. sanity: imports, checkpoint, AMP vs FP32 agreement (~3 min) ----------------
import core                      # fails if the binom_test fix is missing
from DRM_cascade import CascadeRobustModel
m = CascadeRobustModel(SIGMA, mode="cascade", tau=1.01, cheap_ckpt=CKPT)
ds = datasets.CIFAR10("data/cifar10", train=False, transform=__import__("torchvision").transforms.ToTensor())
x = torch.stack([ds[i][0] for i in range(1, 200, 2)]).cuda().repeat_interleave(5, 0)   # not certify images
torch.manual_seed(0); xn = x + torch.randn_like(x) * SIGMA
m.amp = False; p32 = torch.cat([m.expensive_logits(c).argmax(1) for c in xn.split(250)])
m.amp = True;  p16 = torch.cat([m.expensive_logits(c).argmax(1) for c in xn.split(250)])
y = torch.tensor([ds[i][1] for i in range(1, 200, 2)]).cuda().repeat_interleave(5)
print(f"AMP/FP32 agreement {(p32 == p16).float().mean():.4f} | dds acc fp32 {(p32 == y).float().mean():.3f} amp {(p16 == y).float().mean():.3f}")
print(f"cheap acc {(m.cheap_logits(xn).argmax(1) == y).float().mean():.3f}")
# timing of one certify-sized pass (N0+N samples) on one GPU
import time
for amp in [False, True]:
    m.amp = amp; torch.cuda.synchronize(); t0 = time.time()
    for _ in range((N + 100) // BATCH): m.expensive_logits(x[:1].repeat(BATCH, 1, 1, 1) + torch.randn(BATCH, 3, 32, 32, device="cuda") * SIGMA)
    torch.cuda.synchronize(); print(f"amp={amp}: dds ~{time.time() - t0:.1f} s/img on 1 GPU")
del m; torch.cuda.empty_cache()
# -> if agreement < 0.99, set AMP = False in the config cell

In [ ]:
# helper: run certify_cascade on all GPUs in parallel (one shard per GPU), resumable
def certify(mode, tau=None, tag=None):
    tag = tag or mode
    procs = []
    for s in range(NGPU):
        out = f"out/{tag}_{SIG}_s{s}.tsv"
        cmd = (f"CUDA_VISIBLE_DEVICES={s} PYTHONPATH=cifar10 python cifar10/certify_cascade.py --mode {mode} "
               f"--sigma {SIGMA} --cheap_ckpt {CKPT} --skip {SKIP} --max_images {MAX_IMAGES} --N {N} "
               f"--alpha 0.0005 --batch_size {BATCH} --shard {s} --num_shards {NGPU} --resume "
               f"--outfile {out} {AMPF} {'' if tau is None else '--tau ' + str(tau)} "
               f"> logs/{tag}_{SIG}_s{s}.log 2>&1")
        procs.append(subprocess.Popen(cmd, shell=True))
    rc = [p.wait() for p in procs]
    !tail -n 2 logs/{tag}_{SIG}_s*.log
    assert all(r == 0 for r in rc), f"{tag} failed, see logs/"

def report(*extra):
    files = [f"{t}=out/{t}_{SIG}_s*.tsv" for t in ["dds", "cheap"] + list(extra)]
    files = [f for f in files if __import__("glob").glob(f.split("=")[1])]
    fs = " ".join(files)
    !PYTHONPATH=cifar10 python cifar10/compare.py {fs} --intersect

In [ ]:
# ---------------- 3. probe + calibrate tau  (feasibility test, ~15-30 min) ----------------
!PYTHONPATH=cifar10 python cifar10/probe.py --sigma {SIGMA} --cheap_ckpt {CKPT} --num_images 9500 --K 4 \
    --split test_holdout --certify_skip {SKIP} --batch_size {BATCH} {AMPF} --out out/probe_{SIG}.npz
!PYTHONPATH=cifar10 python cifar10/calibrate_tau.py out/probe_{SIG}.npz --delta {DELTA} --alpha {CAL_ALPHA} --out out/tau_{SIG}.txt
TAU = float(open(f"out/tau_{SIG}.txt").read())
print("calibrated tau =", TAU)
# DECISION: accept rate at this tau ~ fraction of diffusion calls saved. < ~0.5  => speed-up < 2x.

In [ ]:
# ---------------- 4. certify: cascade (calibrated tau) + cheap ----------------
TAU = float(open(f"out/tau_{SIG}.txt").read())
certify("cascade", tau=TAU, tag="cascade_cal")
certify("cheap")
report("cascade_cal")

In [ ]:
# ---------------- 5. certify: dds baseline (most expensive; resumable) ----------------
certify("dds")
report("cascade_cal")

In [ ]:
# ---------------- 6. tau sweep for the Pareto plot (optional, run when budget allows) ----------------
for t in [0.5, 0.7, 0.9, 0.99]:
    certify("cascade", tau=t, tag=f"cascade_tau{t}")
report("cascade_cal", *[f"cascade_tau{t}" for t in [0.5, 0.7, 0.9, 0.99]])